In [1]:
import random
from pathlib import Path

from toolz import *
from toolz.curried import *
from toolz.curried.operator import *

import pandas as pd
import pydicom
import mapply

/home/yjeon31/miniconda3/envs/roi-lift/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# NOTE : Please set DATA_PATH to the path where you have stored the raw bcs_dbt
# DATA_PATH = Path("~/DATA/jacob/mammo_tomo/raw/DUKE_DBT/raw/").expanduser()
DATA_PATH = "[path/to/your/raw/bcs_dbt/data]/"

TRAIN_FILE_PATH = DATA_PATH / "file-paths-train.csv"
VALID_PATH = DATA_PATH / "file-paths-validation.csv"
TEST_FILE_PATH = DATA_PATH / "file-paths-test.csv"

TRAIN_BBOX_PATH = DATA_PATH / "boxes-train.csv"
VALID_BBOX_PATH = DATA_PATH / "boxes-validation.csv"
TEST_BBOX_PATH = DATA_PATH / "boxes-test.csv"

SAVE_DIR: Path = Path("../processed/duke/").absolute()

In [3]:
# Merge file paths and bounding boxes from different CSV files
file_path_df = pd.concat(
    [pd.read_csv(file_path) for file_path in [TRAIN_FILE_PATH, VALID_PATH, TEST_FILE_PATH]],
    axis=0,
    ignore_index=True,
)

bbox_df = pd.concat(
    [pd.read_csv(file_path) for file_path in [TRAIN_BBOX_PATH, VALID_BBOX_PATH, TEST_BBOX_PATH]],
    axis=0,
    ignore_index=True,
)

merged_df = pd.merge(bbox_df, file_path_df, on=["PatientID", "StudyUID", "View"], how="left")

In [ ]:
def process_bboxx(row):
    # according to the paper, the depth range is aritificial, taking 1/4 of the volume slices

    x_min = row.X
    y_min = row.Y
    z_center = row.Slice
    width = row.Width
    height = row.Height
    depth = row.VolumeSlices

    x_max = x_min + width
    y_max = y_min + height
    z_min = max(z_center - (depth // 8), 0)
    z_max = min(z_center + (depth // 8), depth - 1)

    return str([z_min, y_min, x_min, z_max, y_max, x_max])


merged_df["rois"] = merged_df.apply(process_bboxx, axis=1)

merged_df["laterality"] = merged_df["View"].apply(lambda x: x[0].upper())
merged_df["view_position"] = merged_df["View"].apply(lambda x: x[1:].upper())

# Add a column with a random unique number for each row
random.seed(0)
merged_df["patch_id"] = [random.getrandbits(25) for _ in range(len(merged_df))]

merged_df["severity"] = merged_df["Class"].apply(lambda x: 1 if x == "cancer" else 0)

merged_df["arch_distortion"] = merged_df["AD"]
merged_df["mass"] = 1 - merged_df["AD"]


def get_shape(row):
    dcm_path_dbt = Path(DATA_PATH).expanduser() / row["descriptive_path"]
    dcm = pydicom.dcmread(dcm_path_dbt)
    d, h, w = dcm.pixel_array.shape
    return pd.Series({"D": d, "H": h, "W": w})


# remove corrupted file
merged_df = merged_df[
    merged_df["descriptive_path"]
    != "Breast-Cancer-Screening-DBT/DBT-P03732/01-01-2000-DBT-S00771-MAMMO screening digital bilateral-09267/21045.000000-59258/1-1.dcm"
]

mapply.init(n_workers=32)
merged_df[["D", "H", "W"]] = merged_df.mapply(get_shape, axis=1)

merged_df = merged_df[
    [
        "PatientID",
        "StudyUID",
        "laterality",
        "view_position",
        "descriptive_path",
        "rois",
        "patch_id",
        "severity",
        "arch_distortion",
        "mass",
        "D",
        "H",
        "W",
    ]
]

merged_df = merged_df.rename(
    columns={
        "PatientID": "patient_id",
        "StudyUID": "study_id",
        "laterality": "laterality",
        "view_position": "view_position",
        "descriptive_path": "dcm_path_dbt",
        "severity": "severity",
        "rois": "rois",
        "patch_id": "patch_id",
        "severity": "severity",
        "arch_distortion": "arch_distortion",
        "mass": "mass",
        "D": "D",
        "H": "H",
        "W": "W",
    }
)

In [ ]:
merged_df = merged_df.rename(
    columns={
        "PatientID": "patient_id",
        "StudyUID": "study_id",
        "laterality": "laterality",
        "view_position": "view_position",
        "descriptive_path": "dcm_path_dbt",
        "severity": "severity",
        "rois": "rois",
        "patch_id": "patch_id",
        "severity": "severity",
        "arch_distortion": "arch_distortion",
        "mass": "mass",
        "D": "D",
        "H": "H",
        "W": "W",
    }
)

In [ ]:
merged_df.to_csv(SAVE_DIR / "meta.csv", index=False)